# N169 · 从暴力到最优的优化阶梯

**目标：** 在每次优化后保留独立可验证的参照。

**先修：** N015, N032, N038, N135, N056。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 重复计算；状态压缩；单调候选；分治统计；操作数；对拍。

**配套讲解来源：** [同名逐章意见](../../comment/169_optimization_ladder_differential_testing.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章的方法论解决一个非常实际的问题：**当你把一个 O(n²) 的暴力算法优化成 O(n log n) 之后，你怎么敢相信新算法是对的？** 答案是：每爬一级优化阶梯，都保留一个"独立可验证的参照"——也就是老实现（暴力解）本身。新旧两个实现喂同样的输入、比对输出，这套流程叫"对拍"（差分测试）。同时，这一章还教你**用"实际操作数"来证明优化真的发生了**，而不是跑一条毫秒曲线就宣称自己优化了。

承载这套方法论的具体题目是"范围和计数"（LeetCode 327）：给你数组 `nums` 和闭区间 `[lower, upper]`，数一数有多少个连续子区间的和落在这个区间里。

具体到数字的小例子：`nums=[-2,5,-1]`，`lower=-2`，`upper=2`。我们把所有连续子区间列出来：

- `[-2]`，和为 -2，在 `[-2,2]` 内，算一个；
- `[-2,5]`，和为 3，超出上界 2，不算；
- `[-2,5,-1]`，和为 2，恰好压着上界，算一个；
- `[5]`，和为 5，不算；
- `[5,-1]`，和为 4，不算；
- `[-1]`，和为 -1，算一个。

所以答案是 **3**。暴力法要枚举全部 6 个区间（n=3 时有 3×4/2=6 个），而优化后的分治法只需要更少的比较次数。这一章就是带你从暴力出发，一级一级爬到分治解，并且全程用对拍和操作数计数来"自证清白"。

## 2. 基础知识：先读懂这些词

- **前缀和（prefix）**：`prefix[j]` 表示前 j 个数的和。有了它，任意区间 `[left, right]` 的和就能写成 `prefix[right+1] - prefix[left]`。我们需要它是因为"数区间和"可以等价转化成"数前缀差"。
- **计数对象（区间对）**：每个区间对应一对位置不同的前缀 `(i, j)`，且 `i < j`（即区间的右端前缀减左端前缀）。注意：**数值重复的前缀不能去重**，因为它们来自不同位置，代表不同的区间。固定好计数对象，后面优化才不会"优化着优化着把题改了"。
- **重复计算**：暴力法里每个区间都从头累加一遍，区间对数是 O(n²) 个；识别出"哪些工作被反复做了"是优化的起点。
- **分治统计**：把前缀数组按原下标切成左右两半，左半边内部、右半边内部递归去数，跨半边的部分单独数——跨半边时所有左半边前缀的下标天然小于右半边前缀的下标，`i < j` 这个约束自动满足。
- **单调候选（两个单向边界）**：跨半边统计时，左前缀 p 越大，合法的右前缀区间 `[p+lower, p+upper]` 整体越靠右。所以两个边界指针 `lo` 和 `hi` 只前进、永不回退，每个右半边元素最多被扫两遍。
- **操作数（operations）**：不跑计时器，而是给算法内部装一个计数器，数它实际做了多少次关键操作。它是"可复现的复杂度证据"，不受机器快慢、Python 版本影响。
- **对拍（差分测试）**：让慢而显然正确的暴力实现和快实现吃完全相同的输入，逐个比对返回值。任何不一致就是快实现有 bug 的铁证。
- **Fenwick（树状数组）**：练习里要求的第三条路线，按数值离散化后用树状数组数"差落在区间里的前缀对"，这里只需知道它是替代解法的名字。

## 3. 思路推导：从小例子开始

优化阶梯一共有四级，我们用 `nums=[-2,5,-1]`、`lower=-2`、`upper=2` 走一遍。

- **Step 1：固定计数对象。** 定义 `prefix=[0, -2, 3, 2]`（开头补一个 0 代表空前缀）。每个区间和就是某对 `i<j` 的 `prefix[j]-prefix[i]`。目标变成：数有多少对 `(i,j)` 满足 `lower <= prefix[j]-prefix[i] <= upper`。
- **Step 2：直接枚举区间对（暴力，O(n²)）。** 手算全部 6 对：`(0,1)` 差 -2 ✓；`(0,2)` 差 3 ✗；`(0,3)` 差 2 ✓；`(1,2)` 差 5 ✗；`(1,3)` 差 4 ✗；`(2,3)` 差 -1 ✓。共 3 对，和第一节数区间的结果一致。这级是"慢但对"，它就是后面的参照。
- **Step 3：前缀和只是省掉求和，还没省掉配对。** 注意即使有了 prefix，两两配对仍是 O(n²) 对。这一级告诉我们瓶颈在"对数"而不在"求和"，所以下一步必须减少配对本身。
- **Step 4：按原下标分治。** 把 `prefix=[0,-2,3,2]` 从中间切开：左半 `[0,-2]`（下标 0,1），右半 `[3,2]`（下标 2,3）。左右内部各自递归；跨半边的配对里，左边任何下标都小于右边任何下标，`i<j` 免检。
- **Step 5：递归内部（以左半 `[0,-2]` 为例）。** 切成 `[0]` 和 `[-2]`。跨这一刀时，左值 0，要求右值 r 满足 `-2 <= r-0 <= 2`，r=-2 满足 ✓。这一层贡献 1，正好对应 Step 2 里的 `(0,1)` 对。右半 `[3,2]` 同理：左值 3、右值 2，差 -1 落在 `[-2,2]` ✓，贡献 1，对应 `(2,3)` 对。
- **Step 6：顶层跨半边统计（单调边界登场）。** 此时左半已排序为 `[-2,0]`，右半已排序为 `[2,3]`。对左值 -2：合法右值区间是 `[-4, 0]`，右半两个都太大，贡献 0（对应 `(1,2)`、`(1,3)` 两对，确实都不合法）。对左值 0：合法右值区间是 `[-2, 2]`，右值 2 ✓、3 ✗，贡献 1（对应 `(0,3)` 对）。注意处理左值 0 时，`lo`、`hi` 都是从左值 -2 用过的位置继续往前走的——这就是"边界不回退"。
- **Step 7：合并三层结果。** 左半 1 + 右半 1 + 跨半边 1 = 3，与暴力一致。分治的排序顺带在合并阶段完成，总代价 O(n log n)。
- **Step 8：对拍 + 数操作。** notebook 里 `benchmark_operation_counts()` 对 n=16,32,64,128,256 各生成一组数，断言两个实现答案相同，并记录暴力的"extensions"（区间扩张次数，恒等于 n(n+1)/2）和分治的"comparisons"（比较次数，随 n log n 增长）。用数字对比代替计时曲线，优化效果一目了然。

## 4. 核心代码：count_range_sum_bruteforce

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def count_range_sum_bruteforce(nums, lower, upper, stats=None):
    if lower > upper:
        raise ValueError('lower must not exceed upper')
    answer = 0
    if stats is not None:
        stats['extensions'] = 0
    for left in range(len(nums)):
        total = 0
        for right in range(left, len(nums)):
            total += nums[right]
            answer += lower <= total <= upper
            if stats is not None:
                stats['extensions'] += 1
    return answer
```

### 逐段读懂代码

### 1. 暴力参照 `count_range_sum_bruteforce`

（对应程序见下方分段实现与完整代码。）

- `if lower>upper: raise ...`：先做输入合法性检查，区间下界大于上界属于调用方犯错，直接抛异常而不是默默返回 0。两个实现都有这行，保证"对拍时输入契约一致"。
- 外层 `for left` 固定区间左端，内层 `for right` 从 `left` 开始向右扩张；`total` 始终维护 `nums[left..right]` 的和，每次只加上新进来的 `nums[right]`，这是"固定左端点递增右端点"的标准暴力写法。
- `answer += lower<=total<=upper`：Python 连续比较直接得到布尔值，True 当 1、False 当 0，命中区间就累加。
- `stats` 是可选的计数器字典：传入时函数会统计 `'extensions'`（区间扩张了多少次），它恒等于区间总数 n(n+1)/2。这就是"操作数"证据的采集口。

### 2. 分治实现 `count_range_sum_merge`

（对应程序见下方分段实现与完整代码。）

- 先构造 `prefix`：`prefix=[0]` 起步（空前缀），逐个累加。注意这里**没有去重也没有排序**，保留原下标顺序，分治切的是"下标"，这一点是正确性的根基。
- 内层 `sort_count(values)` 是"归并排序 + 跨半计数"二合一：`if len(values)<=1` 是递归出口；`middle=len(values)//2` 按位置切半；左右两半递归后各自已排序，返回的 `a`、`b` 是两半内部的命中数，先累进 `answer`。
- 两个 `while` 就是单调候选：对左半的每个 `value`（左半已升序，所以 `value` 递增），`lo` 找到第一个满足 `right[lo]-value>=lower` 的位置（越过的都是"差太小"的），`hi` 找到第一个满足 `right[hi]-value>upper` 的位置（它是第一个"差太大"的）。于是 `[lo, hi)` 这段左闭右开的下标区间里，每个右值都和当前左值配成合法对，`answer += hi-lo`。两个指针在整层循环里只前进不后退，所以跨半计数是线性的。
- 边界条件一严一宽是有讲究的：下界用 `>=lower`（闭区间含端点），上界用 `>upper`（一旦严格大于就停，`hi` 恰好指向第一个不合法的位置）。这一严一宽正好对应闭区间 `[lower, upper]` 的两个端点。
- 最后一段是标准归并：`while i<len(left) and j<len(right)` 把两个有序半合并成一个有序数组返回给上一层，保证父层的 `lo/hi` 单调性能成立。`operations` 用 `nonlocal` 声明，跨半计数和归并里的每次比较都累加进去，最后塞进 `stats['comparisons']`。

### 3. 基准对比 `benchmark_operation_counts`

```python
def benchmark_operation_counts():
    rows = []
    for n in [16, 32, 64, 128, 256]:
        nums = [i * 17 % 11 - 5 for i in range(n)]
        slow = {}
        fast = {}
        a = count_range_sum_bruteforce(nums, -3, 4, slow)
        b = count_range_sum_merge(nums, -3, 4, fast)
        assert a == b
        rows.append((n, slow['extensions'], fast['comparisons']))
    return rows
```

- 测试数据是确定性生成的：`(i*17)%11-5` 让数值在 `[-5,5]` 之间伪随机分布，任何人重跑都得到同一张表。
- 每个规模 n 先跑两个实现，`assert a==b` 是**内嵌的对拍**：基准测试本身先验证正确性，再谈性能。
- 返回的 `rows` 就是 “运行示例”部分 第一张表的内容：n、暴力扩张次数、分治比较次数三列。n=256 时暴力是 256×257/2=32896 次扩张，而分治的比较次数远低于同规模的平方量级，并且增长速度贴合 n log n。

“运行示例”部分 里还有第二张表：`[[-2,5,-1], [-2,2], 3]` 和 `[[0,0], [0,0], 6]`。第二行说的是 `nums=[0,0]`、区间 `[0,0]` 时答案是 6——这来自测试区的 `[0,0,0]` 同款逻辑：三个 0 有 6 个非空连续子区间，全部和为 0，全部命中。它专门提醒你"重复前缀不去重"。

## 5. 分段实现：按顺序运行

**本章接口：** `count_range_sum_bruteforce(nums, lower, upper)`、`count_range_sum_merge(nums, lower, upper)`、`benchmark_operation_counts()`

### count_range_sum_bruteforce

In [1]:
def count_range_sum_bruteforce(nums, lower, upper, stats=None):
    if lower > upper:
        raise ValueError('lower must not exceed upper')
    answer = 0
    if stats is not None:
        stats['extensions'] = 0
    for left in range(len(nums)):
        total = 0
        for right in range(left, len(nums)):
            total += nums[right]
            answer += lower <= total <= upper
            if stats is not None:
                stats['extensions'] += 1
    return answer

### count_range_sum_merge

In [2]:
def count_range_sum_merge(nums, lower, upper, stats=None):
    if lower > upper:
        raise ValueError('lower must not exceed upper')
    prefix = [0]
    for value in nums:
        prefix.append(prefix[-1] + value)
    operations = 0

    def sort_count(values):
        nonlocal operations
        if len(values) <= 1:
            return (values, 0)
        middle = len(values) // 2
        left, a = sort_count(values[:middle])
        right, b = sort_count(values[middle:])
        answer = a + b
        lo = hi = 0
        for value in left:
            while lo < len(right):
                operations += 1
                if right[lo] - value >= lower:
                    break
                lo += 1
            while hi < len(right):
                operations += 1
                if right[hi] - value > upper:
                    break
                hi += 1
            answer += hi - lo
        merged = []
        i = j = 0
        while i < len(left) and j < len(right):
            operations += 1
            if left[i] <= right[j]:
                merged.append(left[i])
                i += 1
            else:
                merged.append(right[j])
                j += 1
        merged.extend(left[i:])
        merged.extend(right[j:])
        return (merged, answer)
    answer = sort_count(prefix)[1]
    if stats is not None:
        stats['comparisons'] = operations
    return answer

### benchmark_operation_counts

In [3]:
def benchmark_operation_counts():
    rows = []
    for n in [16, 32, 64, 128, 256]:
        nums = [i * 17 % 11 - 5 for i in range(n)]
        slow = {}
        fast = {}
        a = count_range_sum_bruteforce(nums, -3, 4, slow)
        b = count_range_sum_merge(nums, -3, 4, fast)
        assert a == b
        rows.append((n, slow['extensions'], fast['comparisons']))
    return rows

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [4]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

show_table(['n','brute extensions','merge comparisons'],benchmark_operation_counts())
show_table(['nums','range','count'],[([-2,5,-1],[-2,2],count_range_sum_merge([-2,5,-1],-2,2)),([0,0],[0,0],count_range_sum_merge([0,0],0,0))])

n,brute extensions,merge comparisons
16,136,141
32,528,359
64,2080,858
128,8256,2042
256,32896,4735


nums,range,count
"[-2, 5, -1]","[-2, 2]",3
"[0, 0]","[0, 0]",3


## 7. 正确性、适用条件与复杂度

每一对前缀恰在最小的、将二者分入不同半边的递归节点被统计一次。右边界条件 `<lower` 和 `<=upper` 分别对应闭区间两端；重复值保留位置重数。分治合并没有遗漏或重复。

**代价：** 递推 T(n)=2T(n/2)+O(n)，时间 O(n log n)，本实现峰值辅助空间 O(n)，递归深度 O(log n)。Python 大整数算术成本还取决于整数位长；这里的 n 复杂度采用字长算术模型。

### 展开理解

**正确性。** 关键问题是"每一对前缀会不会被漏掉或数重"。分治按原下标切半，任意一对下标不同的前缀，必然在某一个递归节点第一次被分到不同半边——而在更深的节点里它们不会再相遇，在更浅的节点里它们还同侧没被配对。所以每一对**恰好在一个节点被统计一次**，不重不漏。跨半计数依赖左右两半各自有序：左值递增推动合法窗口 `[value+lower, value+upper]` 整体右移，`lo/hi` 不回退也不会漏数。至于重复值：两个前缀数值相同但下标不同，它们是不同的对，代码从头到尾按位置处理，天然保留了重数。上下界的写法（`>=lower` 停、`>upper` 停）精确卡住闭区间的两端，差恰好等于 lower 或 upper 都会被计入。

**复杂度。** 每层递归做的事情是"扫一遍左右两半"（跨半计数加归并都是线性），所以递推是 T(n)=2T(n/2)+O(n)，解出来时间 O(n log n)。空间上，`values[:middle]` 这种切片让峰值辅助空间是 O(n)，递归深度是 O(log n)。落到具体规模：n=256 时，暴力要做 32896 次区间扩张，而分治的实测比较次数被 “自动测试”部分 的断言约束在 `10*(n+1)*ceil(log2(n+1))` 以内（约 10×257×8=20560），差距还会随 n 增大继续拉开。还有一个细节：Python 整数不溢出，测试里出现 `10**30` 级别的数时，单次加减的成本取决于位数，所以 notebook 特别注明复杂度采用的是"字长算术模型"（把一次加减当常数）。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的断言分五组，每组目的不同：

1. `assert count_range_sum_merge([-2,5,-1],-2,2)==3`：这是**正常值冒烟测试**，用的就是第三节的经典用例，预期 3 是手算出来的，验证主流程。
2. `assert count_range_sum_merge([0,0,0],0,0)==6`：这是**特殊值测试**——全零数组配上零区间。3 个元素有 6 个非空子区间，全部命中。它专门咬住"重复前缀不能去重"这个坑：如果实现里手滑做了去重，这里会立刻得到 3 而不是 6。
3. `for n in range(6): for nums in itertools.product([-1,0,1],repeat=n): ...`：这是**穷举对拍**，也是本章方法论的灵魂。长度 0 到 5 的所有 `-1/0/1` 数组（共 3^0+...+3^5=364 个），每个配上 4 组区间（含 `lower==upper` 的退化区间和偏一侧的区间），要求分治解与暴力解逐一相等。小输入全空间覆盖，比随机对拍更有说服力。
4. `assert brute_ops==n*(n+1)/2` 和 `assert merge_ops<=10*(n+1)*math.ceil(math.log2(n+1))`：这是**操作数契约**。前者验证暴力扩张次数的解析公式（n=16 时应为 136，n=256 时应为 32896）；后者验证分治比较次数确实被 n log n 量级（放大 10 倍做常数容差）控制住——用数字直接证明"优化真的发生了"。
5. `assert count_range_sum_merge([10**30,-10**30],0,0)==1`：这是**大整数边界测试**。两个天文数字相减后中间区间和恰为 0，命中 1 个（整个数组）。它同时确认实现不依赖数值范围假设、Python 大整数算术下依然正确。

最后一行 `print('N169: 所有本章断言通过')` 只是成功标志；notebook 也提醒过：测试通过不等于一般性证明。

In [5]:
import itertools, math

assert count_range_sum_merge([-2, 5, -1], -2, 2) == 3

assert count_range_sum_merge([0, 0, 0], 0, 0) == 6

for n in range(6):
    for nums in itertools.product([-1, 0, 1], repeat=n):
        for lower, upper in [(-2, 2), (0, 0), (-1, 0), (1, 2)]:
            assert count_range_sum_merge(nums, lower, upper) == count_range_sum_bruteforce(nums, lower, upper)

for n, brute_ops, merge_ops in benchmark_operation_counts():
    assert brute_ops == n * (n + 1) // 2
    assert merge_ops <= 10 * (n + 1) * math.ceil(math.log2(n + 1))

assert count_range_sum_merge([10 ** 30, -10 ** 30], 0, 0) == 1

print('N169: 所有本章断言通过')

N169: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 为范围和同时实现Fenwick替代解。

**练习 2：** 缩减对拍失败样本到最小反例。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（Fenwick 替代解）**：思路方向是"离线 + 离散化"。先把所有会查询的数值（每个 `prefix[j]`、`prefix[j]-upper`、`prefix[j]-lower`）收集起来排序离散化；再从左到右扫描前缀，每到一个 `prefix[j]`，先在树状数组里查询落在 `[prefix[j]-upper, prefix[j]-lower]` 的已插入前缀个数，再把 `prefix[j]` 插入树状数组。写输入输出和边界时，注意 `lower==upper` 与空前缀 `0` 都要参与离散化。验证就用本章的暴力接口做对拍，和 “自动测试”部分 一样先从小数组全排列开始。
- **练习 2（缩减对拍失败样本到最小反例）**：思路方向是"二分/删减法"：拿到一组不一致的输入后，反复尝试删掉一个元素或收紧一个端点，若删除后仍不一致就保留删减，否则恢复该元素，直到任何一步都删不动。手算示例建议从长度 3 左右的反例开始，写清楚两个实现各自的返回值差在哪一对前缀上。这一步的价值在于：最小反例几乎总是直接暴露 bug 的那行代码。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [6]:
# 完整实现：本单元格不依赖其他单元格。
def count_range_sum_bruteforce(nums, lower, upper, stats=None):
    if lower > upper:
        raise ValueError('lower must not exceed upper')
    answer = 0
    if stats is not None:
        stats['extensions'] = 0
    for left in range(len(nums)):
        total = 0
        for right in range(left, len(nums)):
            total += nums[right]
            answer += lower <= total <= upper
            if stats is not None:
                stats['extensions'] += 1
    return answer

def count_range_sum_merge(nums, lower, upper, stats=None):
    if lower > upper:
        raise ValueError('lower must not exceed upper')
    prefix = [0]
    for value in nums:
        prefix.append(prefix[-1] + value)
    operations = 0

    def sort_count(values):
        nonlocal operations
        if len(values) <= 1:
            return (values, 0)
        middle = len(values) // 2
        left, a = sort_count(values[:middle])
        right, b = sort_count(values[middle:])
        answer = a + b
        lo = hi = 0
        for value in left:
            while lo < len(right):
                operations += 1
                if right[lo] - value >= lower:
                    break
                lo += 1
            while hi < len(right):
                operations += 1
                if right[hi] - value > upper:
                    break
                hi += 1
            answer += hi - lo
        merged = []
        i = j = 0
        while i < len(left) and j < len(right):
            operations += 1
            if left[i] <= right[j]:
                merged.append(left[i])
                i += 1
            else:
                merged.append(right[j])
                j += 1
        merged.extend(left[i:])
        merged.extend(right[j:])
        return (merged, answer)
    answer = sort_count(prefix)[1]
    if stats is not None:
        stats['comparisons'] = operations
    return answer

def benchmark_operation_counts():
    rows = []
    for n in [16, 32, 64, 128, 256]:
        nums = [i * 17 % 11 - 5 for i in range(n)]
        slow = {}
        fast = {}
        a = count_range_sum_bruteforce(nums, -3, 4, slow)
        b = count_range_sum_merge(nums, -3, 4, fast)
        assert a == b
        rows.append((n, slow['extensions'], fast['comparisons']))
    return rows

# 示例与回归测试
import itertools, math

assert count_range_sum_merge([-2, 5, -1], -2, 2) == 3

assert count_range_sum_merge([0, 0, 0], 0, 0) == 6

for n in range(6):
    for nums in itertools.product([-1, 0, 1], repeat=n):
        for lower, upper in [(-2, 2), (0, 0), (-1, 0), (1, 2)]:
            assert count_range_sum_merge(nums, lower, upper) == count_range_sum_bruteforce(nums, lower, upper)

for n, brute_ops, merge_ops in benchmark_operation_counts():
    assert brute_ops == n * (n + 1) // 2
    assert merge_ops <= 10 * (n + 1) * math.ceil(math.log2(n + 1))

assert count_range_sum_merge([10 ** 30, -10 ** 30], 0, 0) == 1

print('N169: 所有本章断言通过')

N169: 所有本章断言通过


## 11. 代表题与迁移

- **327. Count of Range Sum（本章主题，canonical）**：这一题就是本章从头到尾推的题，练习"前缀对计数 → 按下标分治 → 单调边界"的完整优化阶梯，暴力解可以直接当对拍参照。
- **239. Sliding Window Maximum（comparison）**：这题练的是同一个"单调候选"思想——窗口滑动时候选最值用单调队列维护、不回头重算，对应本章"边界只进不退"的那一步。它在知识点上与本章互补，用作迁移对照。

notebook 最后提醒：这些题号用于知识映射，本章实现遵循的是教学契约；不要把教学实例当作官方题的完整答案。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。